In [1]:
import sys
import os
import numpy as np
import scipy
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
sys.path.append(os.path.abspath('../library'))
import data as d
import preprocess as p
import utils as u
import bayes as b
import results as r
import figures as fig

In [3]:
mouse_ID = 'C57_913_Qiu'
tau = 0.2 # size of time bin of the data in seconds
Qiu = d.MouseData(mouse_ID, *p.preprocess_data(d.load_data(mouse_ID, tau)), tau)

scipy.io.loadmat failed: Please use HDF reader for matlab v7.3 files, e.g. h5py)
Trying mat73.loadmat...
../datafiles/C57_913_Qiu/time_binned_SpikeInf_200msbins.mat loaded with mat73.loadmat
scipy.io.loadmat failed: Please use HDF reader for matlab v7.3 files, e.g. h5py)
Trying mat73.loadmat...
../datafiles/C57_913_Qiu/time_binned_DiscreteSpikes_200msbins.mat loaded with mat73.loadmat
scipy.io.loadmat failed: Please use HDF reader for matlab v7.3 files, e.g. h5py)
Trying mat73.loadmat...
../datafiles/C57_913_Qiu/target_positions_200msbins.mat loaded with mat73.loadmat
../datafiles/C57_913_Qiu/darktrial_raw.mat loaded with scipy.io.loadmat
scipy.io.loadmat failed: Please use HDF reader for matlab v7.3 files, e.g. h5py)
Trying mat73.loadmat...
../datafiles/C57_913_Qiu/del_trials.mat loaded with mat73.loadmat

Data of: C57_913_Qiu

Spike Probability:
(244, 112, 125)
Trial x 100ms Time Bin x Neuron

Discrete Spikes:
(244, 112, 125)
Trial x 100ms Time Bin x Neuron

Position Matrices:
(244, 

In [4]:
# Intialise variables

Qiu.rewardzone = [46,47,48,49]

Qiu.first5_pos = u.get_firstx_pos(Qiu.position_mtx, x=5)

Qiu.mask = u.create_spikesmask(Qiu.spikes, 
                               Qiu.position_mtx, 
                               spikeprob = Qiu.spikeprob, 
                               rewardzone = Qiu.rewardzone, 
                               firstx_pos = Qiu.first5_pos)

Qiu.position_mtx_masked = u.mask_position_mtx(Qiu.position_mtx, Qiu.rewardzone, Qiu.first5_pos)
Qiu.triallength = u.get_triallength(Qiu.position_mtx_masked)

Qiu.spikes_masked = u.mask_spikes(Qiu.spikes, Qiu.mask)
Qiu.spikes_smoothed = u.gaussiansmooth_spikes(Qiu.spikes, Qiu.mask, sigma=2)
Qiu.firingrate = u.posbinning_data(Qiu.spikes, 'spikes', Qiu.position_mtx_masked, num_pbins=50, tau=0.2)

Qiu.fr_light, Qiu.fr_dark = u.split_lightdark(Qiu.firingrate, Qiu.darktrials)
Qiu.spikes_light, Qiu.spikes_dark = u.split_lightdark(Qiu.spikes_masked, Qiu.darktrials)

Qiu.fr_light_scaled, Qiu.fr_dark_scaled = u.scale_firingrate(Qiu.fr_light, Qiu.fr_dark)

smallest first position: 0.0
largest first position: 34.0
higher in dark:50 | %:0.4
higher in light:75 | %:0.6


/Users/andrewlau/code/bayesian_decoder/library/utils.py:368: RuntimeWarning: invalid value encountered in scalar divide
  firingrate = sum_of_spikes / (occupancy * tau)
/Users/andrewlau/code/bayesian_decoder/library/utils.py:474: RuntimeWarning: divide by zero encountered in scalar divide
  scaling_coefficient_light.append(np.nanmean(fr_dark[:,:,i]) / np.nanmean(fr_light[:,:,i]))
/Users/andrewlau/code/bayesian_decoder/library/utils.py:478: RuntimeWarning: invalid value encountered in multiply
  fr_light_scaled = fr_light * scaling_coefficient_light


In [7]:
posterior_lightlight, decoded_pos_lightlight = b.bayesian_decoder(Qiu, Qiu.fr_light, Qiu.spikes_light, num_pbins=46)
posterior_lightdark, decoded_pos_lightdark = b.bayesian_decoder(Qiu, Qiu.fr_light_scaled, Qiu.spikes_dark, num_pbins=46)
posterior_darkdark, decoded_pos_darkdark = b.bayesian_decoder(Qiu, Qiu.fr_dark, Qiu.spikes_dark, num_pbins=46)
posterior_darklight, decoded_pos_darklight = b.bayesian_decoder(Qiu, Qiu.fr_dark_scaled, Qiu.spikes_light, num_pbins=46)

/Users/andrewlau/code/bayesian_decoder/library/bayes.py:92: RuntimeWarning: Mean of empty slice
  fx = np.nanmean(firingrate, axis=0)        # mean firing rate across trials                                    # 100ms time bin
/Users/andrewlau/code/bayesian_decoder/library/bayes.py:94: RuntimeWarning: divide by zero encountered in log
  log_lamb = np.log(lamb)                    # log lambda
/Users/andrewlau/code/bayesian_decoder/library/bayes.py:110: RuntimeWarning: invalid value encountered in scalar multiply
  log_pnx_before_sum[t,x,i] = (n_i * log_lamb[x,i]) - lamb[x,i] - log_nfac[t,i]
/Users/andrewlau/code/bayesian_decoder/library/bayes.py:130: RuntimeWarning: divide by zero encountered in log
  log_pn = np.log(np.sum((pnx * px), axis=1))
/Users/andrewlau/code/bayesian_decoder/library/bayes.py:136: RuntimeWarning: invalid value encountered in subtract
  posterior[test_trial] = np.exp((log_pnx.T + log_px - log_pn).T)
/Users/andrewlau/code/bayesian_decoder/library/bayes.py:143: Runti

In [8]:
confusion_mtx_lightlight = r.generate_confusion_mtx(decoded_pos_lightlight, 
                                                    Qiu.position_mtx_masked, 
                                                    Qiu.darktrials,
                                                    'light', 
                                                    num_pbins=46)

accuracy_lightlight = r.compute_accuracy(decoded_pos_lightlight, 
                                         Qiu.position_mtx_masked, 
                                         Qiu.darktrials, 
                                         'light')

errors_lightlight, mse_lightlight, rt_mse_lightlight = r.compute_errors(decoded_pos_lightlight, 
                                                                        Qiu.position_mtx_masked, 
                                                                        Qiu.darktrials, 
                                                                        'light')

0.5700325732899023
0.0
number of NaNs in true position: 12810
number of non-NaNs in true position: 3990
total decoder guesses: 3990
accuracy rate excluding NaN true positions: 0.1661654135338346
(3990,)
min error: 0.0 position bins (10cm)
max error: 39.0 position bins (10cm)
mean error: 8.578446115288221 position bins (10cm)

mean squared error: 157.0390977443609
root mean suqred error: 12.531524158870736 position bins (10cm)


In [9]:


confusion_mtx_lightdark = r.generate_confusion_mtx(decoded_pos_lightdark, 
                                                   Qiu.position_mtx_masked, 
                                                   Qiu.darktrials, 
                                                   'dark', 
                                                   num_pbins=46)

accuracy_lightdark = r.compute_accuracy(decoded_pos_lightdark, 
                                        Qiu.position_mtx_masked, 
                                        Qiu.darktrials, 
                                        'dark')

errors_lightdark, mse_lightdark, rt_mse_lightdark = r.compute_errors(decoded_pos_lightdark, 
                                                                     Qiu.position_mtx_masked, 
                                                                     Qiu.darktrials, 
                                                                     'dark')


0.5
0.0
number of NaNs in true position: 7583
number of non-NaNs in true position: 2945
total decoder guesses: 2945
accuracy rate excluding NaN true positions: 0.03599320882852292
(2945,)
min error: 0.0 position bins (10cm)
max error: 39.0 position bins (10cm)
mean error: 11.10220713073005 position bins (10cm)

mean squared error: 196.19626485568762
root mean suqred error: 14.007007705276942 position bins (10cm)


In [10]:

confusion_mtx_darkdark = r.generate_confusion_mtx(decoded_pos_darkdark, 
                                                  Qiu.position_mtx_masked, 
                                                  Qiu.darktrials, 
                                                  'dark', 
                                                  num_pbins=46)

accuracy_darkdark = r.compute_accuracy(decoded_pos_darkdark, 
                                       Qiu.position_mtx_masked, 
                                       Qiu.darktrials, 
                                       'dark')

errors_darkdark, mse_darkdark, rt_mse_darkdark = r.compute_errors(decoded_pos_darkdark, 
                                                                  Qiu.position_mtx_masked, 
                                                                  Qiu.darktrials, 
                                                                  'dark')


0.625
0.0
number of NaNs in true position: 7593
number of non-NaNs in true position: 2935
total decoder guesses: 2935
accuracy rate excluding NaN true positions: 0.03986371379897785
(2935,)
min error: 0.0 position bins (10cm)
max error: 39.0 position bins (10cm)
mean error: 10.634071550255536 position bins (10cm)

mean squared error: 179.5390119250426
root mean suqred error: 13.399216839988917 position bins (10cm)


In [11]:

confusion_mtx_darklight = r.generate_confusion_mtx(decoded_pos_darklight, 
                                                   Qiu.position_mtx_masked, 
                                                   Qiu.darktrials, 
                                                   'light', 
                                                   num_pbins=46)

accuracy_darklight = r.compute_accuracy(decoded_pos_darklight, 
                                        Qiu.position_mtx_masked, 
                                        Qiu.darktrials, 
                                        'light')

errors_darklight, mse_darklight, rt_mse_darklight = r.compute_errors(decoded_pos_darklight, 
                                                                     Qiu.position_mtx_masked, 
                                                                     Qiu.darktrials, 
                                                                     'light')

0.25
0.0
number of NaNs in true position: 12857
number of non-NaNs in true position: 3943
total decoder guesses: 3943
accuracy rate excluding NaN true positions: 0.03347704793304591
(3943,)
min error: 0.0 position bins (10cm)
max error: 39.0 position bins (10cm)
mean error: 10.908191732183617 position bins (10cm)

mean squared error: 187.2135429875729
root mean suqred error: 13.682600008316143 position bins (10cm)
